# Estaturas con Manim · Jupyter / Google Colab

Ejecuta las celdas en orden (**Python 3.11 o posterior**). Las cuatro escenas están incluidas: no necesitas clonar el repositorio.

Las ecuaciones usan **MathTex**, y todos los títulos y explicaciones **Tex**, con la tipografía serif **Latin Modern** de LaTeX. **TCL** significa **teorema central del límite**. El notebook instala y comprueba LaTeX además de Manim.

La película dura unos 54 segundos, sin audio. Por defecto exporta **9:16 vertical, 1080 × 1920 a 30 fps**, con `frame_width = 9` y `frame_height = 16`. Usa `CALIDAD = "vista_previa"` únicamente para pruebas rápidas de menor resolución.

**Parámetros elegidos por el usuario:** media **172 cm**, desviación **7 cm** y umbral **190 cm**. Se mantienen como estimaciones sin atribuirlas a una encuesta específica. A petición del usuario, la aclaración irá en su narración y no aparece como advertencia en pantalla. El resultado normal aproximado es **0,506 %, o 1 de cada 197** bajo el modelo.

**Documentación:** [FUENTES.md](https://github.com/ivanxgallo/manim_estaturas/blob/main/FUENTES.md) distingue estos parámetros del análisis independiente de una copia pública académica de ENS 2016-2017. No se ha validado una cifra publicada de 172 cm y 7 cm. La animación no descarga microdatos.


En Google Colab y Debian/Ubuntu, esta celda instala las dependencias que falten: Cairo/Pango para Manim, **TeX Live + Latin Modern + dvisvgm** para las fórmulas y el texto serif. La primera instalación puede tardar varios minutos.

En Jupyter local se necesitan permisos de administración. En macOS instala `brew install cairo pango pkg-config` y una distribución TeX (por ejemplo MacTeX) que incluya `latex`, `dvisvgm`, `standalone` y `lmodern`. En Windows usa WSL con Ubuntu. **Instalar Manim con pip no instala LaTeX.**


In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Selecciona un kernel con Python 3.11 o posterior.")

paquetes = []
if importlib.util.find_spec("manim") is None:
    pkg_config = shutil.which("pkg-config")
    bibliotecas_listas = bool(pkg_config) and subprocess.run(
        [pkg_config, "--exists", "cairo", "pangocairo"], check=False
    ).returncode == 0
    if not bibliotecas_listas:
        paquetes += ["build-essential", "pkg-config", "libcairo2-dev", "libpango1.0-dev", "ffmpeg"]

# Comprobar LaTeX incluso si Manim ya estaba instalado.
def paquete_tex_disponible(nombre):
    if not shutil.which("kpsewhich"):
        return False
    resultado = subprocess.run(["kpsewhich", nombre], capture_output=True, text=True)
    return resultado.returncode == 0 and bool(resultado.stdout.strip())

latex_listo = all(shutil.which(nombre) for nombre in ("latex", "dvisvgm"))
latex_listo = latex_listo and all(paquete_tex_disponible(nombre) for nombre in
                                ("standalone.cls", "amsmath.sty", "amssymb.sty", "lmodern.sty"))
if not latex_listo:
    paquetes += ["texlive-latex-base", "texlive-latex-extra", "texlive-fonts-recommended", "lmodern", "dvisvgm"]

if paquetes:
    if sys.platform.startswith("linux") and shutil.which("apt-get"):
        comando_manual = "sudo apt-get update && sudo apt-get install -y " + " ".join(paquetes)
        if os.geteuid() == 0:
            prefijo = []
        elif shutil.which("sudo") and subprocess.run(
            ["sudo", "-n", "true"], check=False, capture_output=True
        ).returncode == 0:
            prefijo = ["sudo", "-n"]
        else:
            raise RuntimeError("Ejecuta en una terminal y vuelve a esta celda:\n" + comando_manual)
        entorno = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
        subprocess.run(prefijo + ["apt-get", "update"], check=True, env=entorno)
        subprocess.run(prefijo + ["apt-get", "install", "-y"] + paquetes, check=True, env=entorno)
    else:
        raise RuntimeError("Instala Cairo/Pango y TeX Live/Latin Modern/dvisvgm según las instrucciones de arriba.")

subprocess.run(["latex", "--version"], check=True, capture_output=True)
subprocess.run(["dvisvgm", "--version"], check=True, capture_output=True)
print("Dependencias listas: Manim podrá compilar textos y fórmulas con LaTeX real.")


## 2. Instalar Manim en el mismo Python del notebook

Esta instalación puede tardar unos minutos la primera vez. Si Jupyter solicita reiniciar el kernel, hazlo y vuelve a ejecutar desde el inicio.


In [ ]:
entorno_instalacion = os.environ.copy()
if sys.platform.startswith("linux") and shutil.which("gcc"):
    entorno_instalacion["CC"] = "gcc"
    entorno_instalacion["CXX"] = "g++"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "manim==0.19.0"],
    check=True, env=entorno_instalacion,
)
subprocess.run([sys.executable, "-m", "manim", "--version"], check=True)


## 3. Elegir formato y calidad

`FORMATO`: `"vertical"` para TikTok/Reels, o `"horizontal"`.

`CALIDAD`: `"final"` por defecto para exportar a 1080p y 30 fps. `"vista_previa"` reduce la resolución y es solo para comprobar rápido. El tamaño del reproductor en Jupyter no cambia la resolución del archivo descargado.

Cada ejecución de esta celda crea una carpeta nueva dentro del directorio del notebook para conservar los resultados anteriores.


In [ ]:
from pathlib import Path
import tempfile

FORMATO = "vertical"
CALIDAD = "final"

if FORMATO not in ("vertical", "horizontal"):
    raise ValueError('FORMATO debe ser "vertical" u "horizontal".')
if CALIDAD not in ("vista_previa", "final"):
    raise ValueError('CALIDAD debe ser "vista_previa" o "final".')

vertical = FORMATO == "vertical"
if CALIDAD == "final":
    ancho, alto, fps = (1080, 1920, 30) if vertical else (1920, 1080, 30)
else:
    ancho, alto, fps = (360, 640, 15) if vertical else (854, 480, 15)

CARPETA = Path(tempfile.mkdtemp(prefix="estaturas_jupyter_", dir=Path.cwd())).resolve()
configuracion = f"""[CLI]
pixel_width = {ancho}
pixel_height = {alto}
frame_width = {9 if vertical else 128 / 9}
frame_height = {16 if vertical else 8}
frame_rate = {fps}
media_dir = media
preview = False
"""
(CARPETA / "jupyter.cfg").write_text(configuracion, encoding="utf-8")
print(f"Formato: {FORMATO} · {ancho} × {alto} · {fps} fps")
print(f"Carpeta de salida: {CARPETA}")


## 4. Código completo de las cuatro escenas

Puedes editar `MEDIA`, `DESVIACION`, `UMBRAL` (en centímetros) y `POBLACION` al principio del código de esta celda. Todos los gráficos y cálculos usan esos parámetros. Ejecuta de nuevo esta celda y las siguientes después de cambiarlos.


In [ ]:
codigo = r'''"""Cuatro planos sobre el TCL y una probabilidad de estatura.

Render: python -m manim -ql estaturas.py EstaturasChile
Parámetros proporcionados por el usuario; fuente de encuesta pendiente.
Ver FUENTES.md para distinguirlos del análisis de microdatos ENS 2016-2017.
Requiere LaTeX (TeX Live), Latin Modern y dvisvgm. Sin audio ni archivos externos.
"""

from math import erfc, exp, isfinite, pi, sqrt
import shutil

import numpy as np
from manim import (
    Axes, BLUE_C, Create, DashedLine, DOWN, FadeIn, FadeOut, GOLD,
    GREEN_C, LaggedStart, LEFT, MathTex, MovingCameraScene,
    Rectangle, RIGHT, RoundedRectangle, Tex, TexTemplate,
    Transform, UP, VGroup, WHITE, Write, config,
)


# Centímetros. Valores proporcionados por el usuario; encuesta aún sin identificar.
# No atribuir estos valores a ENS 2016-2017: los de la copia analizada son distintos.
MEDIA = 172.0
DESVIACION = 7.0
UMBRAL = 190.0
POBLACION = "Hombres en Chile"
SEMILLA = 2026
FONDO = "#101827"
TENUE = "#A9B8CC"

# Latin Modern es la versión vectorial de la tipografía de Computer Modern.
# El texto y las fórmulas pasan por el mismo compilador LaTeX y la misma plantilla.
PLANTILLA_TEX = TexTemplate()
PLANTILLA_TEX.add_to_preamble(r"\usepackage[utf8]{inputenc}")
PLANTILLA_TEX.add_to_preamble(r"\usepackage[T1]{fontenc}")
PLANTILLA_TEX.add_to_preamble(r"\usepackage{lmodern}")


def probabilidad_superior(umbral=UMBRAL, media=MEDIA, desviacion=DESVIACION):
    """Cola normal estable, sin restar de 1 una CDF próxima a 1."""
    if not all(isfinite(x) for x in (umbral, media, desviacion)):
        raise ValueError("Los parámetros deben ser finitos.")
    if desviacion <= 0:
        raise ValueError("La desviación estándar debe ser positiva.")
    return 0.5 * erfc((umbral - media) / (desviacion * sqrt(2)))


def decimal(valor, lugares=2):
    return f"{valor:.{lugares}f}".replace(".", ",")


def texto(contenido, tam=28, color=WHITE):
    """Prosa serif compilada con LaTeX; el contenido se trata como texto literal."""
    escapes = {
        "\\": r"\textbackslash{}", "{": r"\{", "}": r"\}",
        "$": r"\$", "&": r"\&", "%": r"\%", "#": r"\#", "_": r"\_",
        "^": r"\textasciicircum{}", "~": r"\textasciitilde{}",
        "\n": r"\\", "·": r"\textperiodcentered{}", "Φ": r"\(\Phi\)",
    }
    literal = "".join(escapes.get(caracter, caracter) for caracter in contenido)
    return Tex(literal, tex_template=PLANTILLA_TEX, font_size=tam * 1.45, color=color)


def ecuacion(contenido, tam=32, color=WHITE, ancho=None):
    """Matemática LaTeX real, con fracciones, raíces y subíndices tipográficos."""
    formula = MathTex(contenido, tex_template=PLANTILLA_TEX, font_size=tam * 1.3, color=color)
    if ancho is not None and formula.width > ancho:
        formula.scale_to_fit_width(ancho)
    return formula


def numero_tex(valor, lugares=None):
    cadena = f"{valor:g}" if lugares is None else f"{valor:.{lugares}f}"
    return cadena.replace(".", "{,}")


def normal(x, media=0.0, desviacion=1.0):
    return exp(-0.5 * ((x - media) / desviacion) ** 2) / (desviacion * sqrt(2 * pi))


class PlanoBase(MovingCameraScene):
    def setup(self):
        super().setup()
        self.camera.background_color = FONDO
        self.vertical = config.frame_height > config.frame_width
        faltantes = [nombre for nombre in ("latex", "dvisvgm") if not shutil.which(nombre)]
        if faltantes:
            raise RuntimeError(
                f"Falta instalar: {', '.join(faltantes)}. Ejecuta la primera celda del "
                "notebook actualizado o consulta las dependencias LaTeX en README.md."
            )
        # Validar también cuando se usa únicamente el plano introductorio.
        probabilidad_superior()

    def cabecera(self, numero, titulo, subtitulo):
        if self.vertical:
            titulos = {
                1: "Teorema central\ndel límite (TCL)",
                2: "Un modelo para\nlas estaturas",
                3: f"¿Qué tan probable es\nsuperar {UMBRAL:g} cm?",
                4: "De una cola\na una probabilidad",
            }
            etiqueta = texto(f"{numero:02d}  /  ESTATURAS", 22, BLUE_C)
            titulo_m = texto(titulos[numero], 36)
            if titulo_m.width > 7.4:
                titulo_m.scale_to_fit_width(7.4)
            if subtitulo == POBLACION:
                subtitulo = POBLACION.replace(" · ", "\n")
            else:
                subtitulo = "De muchas contribuciones\na una campana"
            sub = texto(subtitulo, 24, TENUE)
            if sub.width > 7.4:
                sub.scale_to_fit_width(7.4)
            grupo = VGroup(etiqueta, titulo_m, sub).arrange(DOWN, aligned_edge=LEFT, buff=0.23)
            grupo.move_to(LEFT * 0.25).set_y(6.3 - grupo.height / 2)
            self.play(FadeIn(grupo, shift=DOWN * 0.15))
            return
        etiqueta = texto(f"{numero:02d}  /  ESTATURAS", 18, BLUE_C).to_corner(UP + LEFT)
        titulo_m = texto(titulo, 37).next_to(etiqueta, DOWN, aligned_edge=LEFT, buff=0.2)
        sub = texto(subtitulo, 21, TENUE).next_to(titulo_m, DOWN, aligned_edge=LEFT, buff=0.16)
        grupo = VGroup(etiqueta, titulo_m, sub)
        if grupo.width > 13.1:
            grupo.scale_to_fit_width(13.1).to_corner(UP + LEFT)
        self.play(FadeIn(grupo, shift=DOWN * 0.15))

    def limpiar(self):
        self.play(*[FadeOut(m) for m in list(self.mobjects)], run_time=0.6)

    def ejes_alturas(self):
        # Incluir siempre el umbral y al menos cuatro desviaciones por lado.
        minimo = 10 * np.floor(min(MEDIA - 4 * DESVIACION, UMBRAL - DESVIACION) / 10)
        maximo = 10 * np.ceil(max(MEDIA + 4 * DESVIACION, UMBRAL + DESVIACION) / 10)
        altura = normal(MEDIA, MEDIA, DESVIACION)
        ejes = Axes(
            x_range=[minimo, maximo, 10], y_range=[0, altura * 1.2, altura / 2],
            x_length=7.0 if self.vertical else 10.8,
            y_length=3.7 if self.vertical else 3.1, tips=False,
            axis_config={"color": TENUE, "include_ticks": False},
        )
        if self.vertical:
            ejes.shift(LEFT * 0.25)
        etiquetas = VGroup(*[
            ecuacion(numero_tex(x), 22 if self.vertical else 20, TENUE).next_to(ejes.c2p(x, 0), DOWN, buff=0.15)
            for x in np.arange(minimo, maximo + 1, 10)
        ])
        unidad = texto("Estatura (cm)", 24 if self.vertical else 20, TENUE).next_to(etiquetas, DOWN, buff=0.18)
        if self.vertical:
            densidad = texto("Densidad", 21, TENUE).next_to(ejes, UP, buff=0.5).align_to(ejes, LEFT)
        else:
            densidad = texto("Densidad", 18, TENUE).next_to(ejes, LEFT, buff=0.12).rotate(pi / 2)
        curva = ejes.plot(lambda x: normal(x, MEDIA, DESVIACION), color=BLUE_C)
        return ejes, VGroup(etiquetas, unidad, densidad), curva

    def plano_tcl(self):
        self.cabecera(1, "Teorema central del límite (TCL)",
                      "De muchas contribuciones a una campana")
        ejes = Axes(x_range=[-4, 4, 1], y_range=[0, 0.65, 0.2],
                    x_length=7.0 if self.vertical else 10.7,
                    y_length=3.7 if self.vertical else 3.0, tips=False,
                    axis_config={"include_ticks": False, "color": TENUE}).shift(DOWN * 0.25)
        if self.vertical:
            ejes.shift(LEFT * 0.25)
        etiquetas = VGroup(*[
            ecuacion(str(x), 20, TENUE).next_to(ejes.c2p(x, 0), DOWN, buff=0.12)
            for x in range(-3, 4)
        ])
        formula_tcl = ecuacion(r"Z_n=\frac{X_1+\cdots+X_n-n\mu}{\sigma\sqrt{n}}", 32).move_to(DOWN * 2.7)
        condicion = texto("Variables i.i.d. · varianza finita y positiva", 20, TENUE).move_to(DOWN * 3.55)
        if self.vertical:
            formula_tcl = ecuacion(r"Z_n=\frac{X_1+\cdots+X_n-n\mu}{\sigma\sqrt{n}}", 34, ancho=7.3).move_to(LEFT * 0.25 + DOWN * 3.25)
            condicion = texto("Variables independientes\ne idénticamente distribuidas\nVarianza finita y positiva", 23, TENUE).move_to(LEFT * 0.25 + DOWN * 4.65)
        self.play(Create(ejes), FadeIn(etiquetas), Write(formula_tcl), FadeIn(condicion))
        referencia = ejes.plot(normal, color=GOLD, stroke_width=3)
        leyenda = texto("Referencia: normal estándar", 19, GOLD).move_to(RIGHT * 3.5 + UP * 1.35)
        if self.vertical:
            leyenda = texto("Referencia: normal estándar", 22, GOLD).move_to(LEFT * 0.25 + UP * 2.5)
        self.play(Create(referencia), FadeIn(leyenda))
        rng = np.random.default_rng(SEMILLA)
        muestras = rng.uniform(-0.5, 0.5, size=(40000, 40))
        acumuladas = muestras.cumsum(axis=1)
        bordes = np.linspace(-4, 4, 49)

        def histograma(n):
            valores = acumuladas[:, n - 1] / sqrt(n / 12)
            # Dividir por la muestra completa, sin renormalizar al rango visible.
            cuentas, _ = np.histogram(valores, bins=bordes)
            densidades = cuentas / (len(valores) * np.diff(bordes))
            barras = VGroup()
            for inicio, fin, densidad in zip(bordes[:-1], bordes[1:], densidades):
                barra = Rectangle(
                    width=(ejes.c2p(fin, 0) - ejes.c2p(inicio, 0))[0] * 0.94,
                    height=max(float(densidad) * ejes.y_axis.unit_size, 0.001),
                    stroke_width=0, fill_color=BLUE_C, fill_opacity=0.65,
                ).move_to(ejes.c2p((inicio + fin) / 2, float(densidad) / 2))
                barras.add(barra)
            return barras

        barras = histograma(1)
        contador = ecuacion(r"n=1\quad\text{contribución}", 26, BLUE_C).move_to(LEFT * 3.2 + UP * 1.35)
        if self.vertical:
            contador.move_to(LEFT * 0.25 + UP * 3.05)
        self.play(FadeIn(barras), FadeIn(contador))
        self.wait(1.5)
        for n in (2, 5, 15, 40):
            nuevo = ecuacion(rf"n={n}\quad\text{{contribuciones}}", 26, BLUE_C).move_to(contador)
            self.play(Transform(barras, histograma(n)), Transform(contador, nuevo), run_time=1.1)
            self.wait(1)
        self.wait(1.5)

    def plano_estaturas(self):
        self.cabecera(2, "Un modelo para las estaturas", POBLACION)
        ejes, etiquetas, curva = self.ejes_alturas()
        self.play(Create(ejes), FadeIn(etiquetas))
        self.play(Create(curva), run_time=1.5)
        marcas = VGroup()
        for x, etiqueta, color in (
            (MEDIA - DESVIACION, r"\mu-\sigma", TENUE),
            (MEDIA, r"\mu", GOLD),
            (MEDIA + DESVIACION, r"\mu+\sigma", TENUE),
        ):
            linea = DashedLine(ejes.c2p(x, 0), ejes.c2p(x, normal(x, MEDIA, DESVIACION)), color=color)
            rotulo = ecuacion(etiqueta, 25, color).next_to(linea, UP, buff=0.1)
            marcas.add(VGroup(linea, rotulo))
        self.play(LaggedStart(*[FadeIn(m) for m in marcas], lag_ratio=0.35))
        parametros = ecuacion(rf"\mu={numero_tex(MEDIA)}\,\mathrm{{cm}}\qquad\sigma={numero_tex(DESVIACION)}\,\mathrm{{cm}}", 30, GOLD).move_to(DOWN * 2.65)
        detalle = texto("La media fija el centro; la desviación estándar mide la dispersión.", 20, TENUE).move_to(DOWN * 3.15)
        if self.vertical:
            parametros = ecuacion(rf"\begin{{gathered}}\mu={numero_tex(MEDIA)}\,\mathrm{{cm}}\\\sigma={numero_tex(DESVIACION)}\,\mathrm{{cm}}\end{{gathered}}", 35, GOLD).move_to(LEFT * 0.25 + DOWN * 3.5)
            detalle = texto("La media fija el centro.\nLa desviación estándar\nmide la dispersión.", 23, TENUE).move_to(LEFT * 0.25 + DOWN * 4.8)
        self.play(Write(parametros), FadeIn(detalle))
        self.wait(4)

    def plano_cola(self):
        self.cabecera(3, f"¿Cuál es la probabilidad de superar {UMBRAL:g} cm?", POBLACION)
        ejes, etiquetas, curva = self.ejes_alturas()
        self.play(Create(ejes), FadeIn(etiquetas), Create(curva))
        extremo = ejes.x_range[1]
        area = ejes.get_area(curva, x_range=(UMBRAL, extremo), color=GREEN_C, opacity=0.85)
        linea = DashedLine(ejes.c2p(UMBRAL, 0), ejes.c2p(UMBRAL, normal(MEDIA, MEDIA, DESVIACION) * 0.85), color=GREEN_C)
        umbral = ecuacion(rf"{numero_tex(UMBRAL)}\,\mathrm{{cm}}", 25, GREEN_C).next_to(linea, UP, buff=0.12)
        self.play(Create(linea), FadeIn(umbral), FadeIn(area), run_time=1.5)
        self.camera.frame.save_state()
        self.play(self.camera.frame.animate.scale(0.48).move_to(ejes.c2p(UMBRAL + DESVIACION * 0.45, normal(MEDIA, MEDIA, DESVIACION) * 0.16)), run_time=1.5)
        self.wait(2)
        self.play(self.camera.frame.animate.restore(), run_time=1.5)
        formula = ecuacion(rf"P(H>{numero_tex(UMBRAL)})=1-\Phi\!\left(\frac{{{numero_tex(UMBRAL)}-\mu}}{{\sigma}}\right)", 29).move_to(DOWN * 2.9)
        aclaracion = texto("Φ: función de distribución de la normal estándar · área total = 1", 18, TENUE).move_to(DOWN * 3.65)
        if self.vertical:
            formula = ecuacion(rf"\begin{{gathered}}P(H>{numero_tex(UMBRAL)})\\=1-\Phi\!\left(\frac{{{numero_tex(UMBRAL)}-\mu}}{{\sigma}}\right)\end{{gathered}}", 33, ancho=7.3).move_to(LEFT * 0.25 + DOWN * 3.85)
            aclaracion = texto("Φ: distribución acumulada\nde la normal estándar\nEl área total es 1", 23, TENUE).move_to(LEFT * 0.25 + DOWN * 5.35)
        self.play(Write(formula), FadeIn(aclaracion))
        self.wait(3)

    def plano_resultado(self):
        self.cabecera(4, "De una cola a una probabilidad", POBLACION)
        z = (UMBRAL - MEDIA) / DESVIACION
        p = probabilidad_superior()
        cuenta = ecuacion(rf"z=\frac{{{numero_tex(UMBRAL)}-{numero_tex(MEDIA)}}}{{{numero_tex(DESVIACION)}}}\approx {numero_tex(z, 2)}", 34).move_to(UP * 0.85)
        porcentaje = ecuacion(rf"P(H>{numero_tex(UMBRAL)})\approx {numero_tex(p * 100, 2)}\,\%", 45, GREEN_C).move_to(DOWN * 0.15)
        equivalencia = texto(f"Aproximadamente 1 de cada {round(1 / p):,} personas".replace(",", "."), 29).move_to(DOWN * 1.15) if p > 0 else texto("Cola por debajo de la precisión numérica", 26)
        marco = RoundedRectangle(width=12.0, height=3.5, corner_radius=0.2,
                                 stroke_color=BLUE_C, stroke_opacity=0.45).move_to(DOWN * 0.15)
        if self.vertical:
            cuenta = ecuacion(rf"z=\frac{{{numero_tex(UMBRAL)}-{numero_tex(MEDIA)}}}{{{numero_tex(DESVIACION)}}}\approx {numero_tex(z, 2)}", 34, ancho=7.0).move_to(LEFT * 0.25 + UP * 1.6)
            porcentaje = ecuacion(rf"\approx {numero_tex(p * 100, 2)}\,\%", 65, GREEN_C).move_to(LEFT * 0.25)
            encabezado = ecuacion(rf"P(H>{numero_tex(UMBRAL)}\,\mathrm{{cm}})", 30, GREEN_C).next_to(porcentaje, UP, buff=0.2)
            porcentaje = VGroup(porcentaje, encabezado)
            equivalencia = texto(f"Aproximadamente\n1 de cada {round(1 / p):,} personas".replace(",", "."), 30).move_to(LEFT * 0.25 + DOWN * 1.6) if p > 0 else texto("Cola por debajo de\nla precisión numérica", 26).move_to(DOWN * 1.6)
            marco = RoundedRectangle(width=7.5, height=6, corner_radius=0.2,
                                     stroke_color=BLUE_C, stroke_opacity=0.45).move_to(LEFT * 0.25)
        self.play(Create(marco), Write(cuenta))
        self.play(Write(porcentaje), run_time=1.5)
        self.play(FadeIn(equivalencia, shift=UP * 0.15))
        nota = VGroup(
            texto(f"Probabilidad de superar {UMBRAL:g} cm.", 22, GOLD),
            texto("Área bajo la cola derecha de la curva normal.", 20, TENUE),
        ).arrange(DOWN, buff=0.18).move_to(DOWN * 2.65)
        if self.vertical:
            nota = VGroup(
                texto(f"Probabilidad de superar\n{UMBRAL:g} cm.", 25, GOLD),
                texto("Área bajo la cola derecha\nde la curva normal.", 23, TENUE),
            ).arrange(DOWN, buff=0.3).move_to(LEFT * 0.25 + DOWN * 4.3)
        self.play(FadeIn(nota))
        self.wait(5)


class TeoremaCentralLimite(PlanoBase):
    def construct(self):
        self.plano_tcl()


class DistribucionEstaturas(PlanoBase):
    def construct(self):
        self.plano_estaturas()


class ColaSobre190(PlanoBase):
    def construct(self):
        self.plano_cola()


class ResultadoProbabilidad(PlanoBase):
    def construct(self):
        self.plano_resultado()


class EstaturasChile(PlanoBase):
    """Película completa; los planos también pueden renderizarse por separado."""

    def construct(self):
        for indice, plano in enumerate((self.plano_tcl, self.plano_estaturas,
                                        self.plano_cola, self.plano_resultado)):
            if indice:
                self.limpiar()
            plano()
'''

archivo_escenas = CARPETA / "estaturas.py"
archivo_escenas.write_text(codigo, encoding="utf-8")
print(f"Código guardado en: {archivo_escenas}")


## 5. Renderizar

Por defecto se renderiza la película completa. Para un plano concreto, cambia `ESCENA` por `TeoremaCentralLimite`, `DistribucionEstaturas`, `ColaSobre190` o `ResultadoProbabilidad`.

Espera a que termine: el video aparece en la celda siguiente. La calidad final requiere más tiempo.


In [ ]:
ESCENA = "EstaturasChile"
NOMBRE_VIDEO = f"{ESCENA}_{FORMATO}_{CALIDAD}"
comando = [
    sys.executable, "-m", "manim",
    "--config_file", "jupyter.cfg", "--progress_bar", "none",
    "estaturas.py", ESCENA, "-o", NOMBRE_VIDEO,
]
subprocess.run(comando, cwd=CARPETA, check=True)
video = CARPETA / "media" / "videos" / "estaturas" / f"{alto}p{fps}" / f"{NOMBRE_VIDEO}.mp4"
if not video.is_file() or video.stat().st_size == 0:
    raise RuntimeError(f"No se encontró un video válido en {video}")
print(f"Video generado: {video} ({video.stat().st_size / 1_000_000:.2f} MB)")


## 6. Ver y descargar el video

El reproductor incrusta el MP4 en la salida de Jupyter. Para descargarlo, usa el enlace (Jupyter local) o ejecuta la última celda (Google Colab).


In [ ]:
from IPython.display import FileLink, Video, display

display(Video(str(video), embed=True, width=360 if vertical else 800))
display(FileLink(str(video.relative_to(Path.cwd())), result_html_prefix="Descargar MP4: "))


In [ ]:
# Opcional: descarga directa si estás usando Google Colab.
try:
    from google.colab import files
except ImportError:
    print("En Jupyter, usa el enlace Descargar MP4 de la celda anterior.")
else:
    files.download(str(video))
